# Qualitative interpretation of intervention sizes

What constitutes a 'small' or 'big' intervention when talking about adjustments to baseline activation?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from ising.model import UpdateMethod
from matplotlib.lines import Line2D
from scipy.special import expit

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202606081503

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)
_, Y, X = dataset.indices_to_numpy(kind="time-series", binarise=True, seed=RANDOM_SEED)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
model = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)
cov_model = Ising.fit(
    Y,
    X,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)
# model.adj[abs(model.j) < 0.125] = 0

### Estimate probability of activation for each individual, on each spin

In [ ]:
sns.displot(model.pcond(1, prev=Y[:, -1], spin=3) * 2 - 1)

## Figures

1. How does the change in baseline activtion compare to 1?

    1. Show logistic plot and values of baseline activations (for model not using covariates). How does delta shift change the baseline activation?

    2. For model using covariates, show distribution of baseline activations for CC anthropogenic. Show what delta shift look like for different individuals.

2. How does it compare to cumulative effect of other variables?

    1. For non-covariate model: show how conditional probability changes for that node after intervention.

    2. For covariate model: show how conditional probability changes after intervention for different types of individuals (based on baseline for this spin; can choose values based on the distribution from 1.)

### 1A: Logistic plot; baseline activations 

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.5), constrained_layout=True)

# Plot relationship between baseline activation and activation probability given no
#   interactions
h = np.linspace(-6, 6, 100)
p = expit(h)
ax.plot(h, p, color="black", zorder=5, linewidth=2, clip_on=False)

# Plot change in activation probability for different sizes of interventions, given
#   initial baseline activation
model = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)

ax.plot(
    [model.h[1], model.h[1]],
    [0, expit(model.h[1])],
    color="grey",
    linestyle="dashed",
    linewidth=1,
)
ax.plot(
    [-6, model.h[1]],
    [expit(model.h[1]), expit(model.h[1])],
    color="grey",
    linestyle="dashed",
    linewidth=1,
)

for delta, colour in zip(
    (0.5, 1.5, 2.5), ("#7BAFDE", "#5289C7", "#1965B0"), strict=True
):
    _h = model.h[1] + delta
    ax.plot(
        [_h, _h],
        [0, expit(_h)],
        color=colour,
        linestyle="dashed",
        linewidth=1,
        label=f"$\\delta h = {delta:.1f}$",
    )
    ax.plot(
        [-6, _h], [expit(_h), expit(_h)], color=colour, linestyle="dashed", linewidth=1
    )

ax.set_xlim(-6, 6)
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlabel("$h$")
ax.set_ylabel("$P(S_i = +1)$")

ax.legend(
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
    handlelength=1.5,
)

fig.savefig(
    "../meeting_6_11/qual_int_level_logistic_activation_prob_diff_no_interactions_cc_anthro.pdf",
    bbox_inches="tight",
)

### 1B: Covariate model; distribution of baseline activations and effects of delta shifts

In [ ]:
model = Ising.fit(
    Y,
    X,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)

fig, axes = plt.subplots(ncols=2, figsize=(5.5, 2), constrained_layout=True)

# Distribution of baseline activations
h = model.h[1, 0] + model.h[1, 1:] @ X[:, -1].T
sns.histplot(h, ax=axes[0], stat="density")
axes[0].set_xlabel(r"$h_\text{anthropogenic}$")


# Distributions of activation probability under different interventions
sns.kdeplot(expit(h), ax=axes[1], color="grey")
axes[1].set_xlabel(r"$P(S_i = +1)$")
for delta, colour in zip(
    (0.5, 1.5, 2.5), ("#7BAFDE", "#5289C7", "#1965B0"), strict=True
):
    _h = h + delta
    sns.kdeplot(expit(_h), ax=axes[1], color=colour)

axes[1].set_ylabel(None)

labels = [f"$\\delta h = {delta:.1f}$" for delta in (0.5, 1.5, 2.5)]
handles = [
    Line2D([0], [0], color="#7BAFDE", lw=2),
    Line2D([0], [0], color="#5289C7", lw=2),
    Line2D([0], [0], color="#1965B0", lw=2),
]

axes[1].legend(
    handles,
    labels,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
    handlelength=1,
    columnspacing=1,
)

for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Individual baseline activation")
fig.savefig(
    "../meeting_6_11/qual_int_level_activation_prob_diff_covariates_no_interactions_cc_anthro.pdf",
    bbox_inches="tight",
)

In [ ]:
p1 = 0.8
p2 = 0.97

np.log(p2 / (1 - p2)) - np.log(p1 / (1 - p1))

### 2A: How does intervention change conditional activation probability

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(5.7, 2), constrained_layout=True)

model = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)

p = np.empty((Y.shape[0], 3), dtype=np.float64)
p[:, 0] = model.pcond(1, prev=Y[:, -1], spin=1)

for i, delta in enumerate((0.5, 1.5), start=1):
    imodel = model.intervene(
        spins=np.array([1]), field_offset=np.array([delta]), seed=RANDOM_SEED
    )
    p[:, i] = imodel.pcond(1, prev=Y[:, -1], spin=1)

# Plot original conditional activation probability for 'CC anthropogenic'
sns.kdeplot(p[:, 0], ax=axes[0], clip=(0, 1), fill=True, color="grey")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel(r"$P(S_i = +1)$")
axes[0].set_title("Individual activation probability", pad=25)


# Plot ECDFs for each intervention
for i, colour in enumerate(("grey", "#7BAFDE", "#5289C7")):
    sns.ecdfplot(p[:, i], ax=axes[1], clip_on=False, color=colour)

labels = ["Baseline"] + [f"$\\delta h = {delta:.1f}$" for delta in (0.5, 1.5)]
handles = [
    Line2D([0], [0], color="grey", lw=2),
    Line2D([0], [0], color="#7BAFDE", lw=2),
    Line2D([0], [0], color="#5289C7", lw=2),
]

axes[1].legend(
    handles,
    labels,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
    handlelength=1,
    columnspacing=1,
)
axes[1].set_xlabel(r"$P(S_i = +1)$")
axes[1].set_xlim(0, 1)
axes[1].set_title("ECDF", pad=25)

for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.savefig(
    "../meeting_6_11/qual_int_level_activation_prob_diff_no_covariates_cc_anthro.pdf",
    bbox_inches="tight",
)

### 2B: Change in conditional probability for different baselines

Take baselines $h \in \{0.5, 0.7\}$ from **1B**.

In [ ]:
model = Ising.fit(
    Y,
    X,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)

In [ ]:
eff_h = model.h[1, 0] + model.h[1, 1:] @ X[:, -1].T
gA = eff_h < 0.5
gB = (eff_h > 0.5) & (eff_h < 0.8)

imodels = [
    model.intervene(
        spins=np.array([1]),
        field_offset=np.array([[delta, 0, 0, 0, 0, 0, 0]]),
        seed=RANDOM_SEED,
    )
    for delta in (0.5, 1.5)
]

for group in (gA, gB):
    baseline = model.pcond(1, prev=Y[group, -1], X=X[group, -1], spin=1)

    for imodel in imodels:
        intervene_p = imodel.pcond(1, prev=Y[group, -1], X=X[group, -1], spin=1)
        diff = intervene_p - baseline
        # break
        sns.displot(diff)
    # break

### 3: How does intervention size compare to the cumulative influence on spin $i$?

No covariates

In [ ]:
model = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    node_labels=dataset.schema.get_short_names("measurement"),
    self_loops=True,
)

In [ ]:
model.node_labels

In [ ]:
np.abs(model.j).sum(axis=0)

In [ ]:
abs(model.j[:, 1]).sum()

In [ ]:
model.node_labels

In [ ]:
sample_individuals = rng.choice(np.arange(Y.shape[0]), size=10, replace=False)
sample_Y, sample_X = Y[sample_individuals, -1], X[sample_individuals, -1]

In [ ]:
model.h[1, 0] + model.h[1, 1:] @ sample_X[0]

In [ ]:
model.draw_state(sample_Y[0], seed=20260615)

In [ ]:
baseline_p = expit(model.h)
small_intervention_p = expit(model.h + 1)
med_intervention_p = expit(model.h + 2)
big_intervention_p = expit(model.h + 3)

plot_df = pl.DataFrame(
    {
        "Variable": model.node_labels,
        "None": baseline_p,
        "Small intervention": small_intervention_p,
        "Medium intervention": med_intervention_p,
        "Large intervention": big_intervention_p,
    }
).unpivot(index="Variable", variable_name="Intervention", value_name="Probability")

sns.barplot(plot_df, x="Variable", y="Probability", hue="Intervention")